In [16]:
import time
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

DATA_PATH = "/kaggle/input/datasets/shauryapandey19/pm25-dataset/delhi_33station_pm25_spatiotemporal_dataset_2018_2026.csv"

# only the columns the model uses, keeps RAM and load time down on ~2.5M rows
use_columns = [
    "timestamp_ist", "station_name", "latitude", "longitude", "pm25", "pm25_original",
    "wind_speed_m_s", "wind_dir_sin", "wind_dir_cos", "boundary_layer_height_m",
    "month", "fire_frp_sum", "fire_count",
]

df = pd.read_csv(DATA_PATH, usecols=use_columns, parse_dates=["timestamp_ist"])

df = df.rename(columns={"timestamp_ist": "timestamp"})
df["station_id"] = df["station_name"]
df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

print(df.shape)
print(df["station_id"].nunique())

# must be 0, the pivot in cell 4 needs unique (station, hour) pairs
print(df.duplicated(subset=["station_id", "timestamp"]).sum())

(2392598, 14)
33
0


In [8]:
# prep already set PM2.5 <= 0 to NaN before interpolation, so this only verifies it
print("pm25_original <= 0:", int((df["pm25_original"] <= 0).sum()))
print("pm25 <= 0:", int((df["pm25"] <= 0).sum()), "| pm25 nulls:", int(df["pm25"].isna().sum()))
print("pm25_original nulls (interpolated hours):", int(df["pm25_original"].isna().sum()))

assert df["pm25"].notna().all() and (df["pm25"] > 0).all()
assert not (df["pm25_original"] <= 0).any()  # NaN compares False, so nulls pass

pm25_original <= 0: 0
pm25 <= 0: 0 | pm25 nulls: 0
pm25_original nulls (interpolated hours): 25674


In [9]:
df["ist_date"] = df["timestamp"].dt.normalize()

# fire columns are one value per (station, IST day), repeated on every hour of that day
fire_daily_columns = ["fire_frp_sum", "fire_count"]
assert df.groupby(["station_id", "ist_date"])[fire_daily_columns].nunique().to_numpy().max() == 1

daily_fire = df.drop_duplicates(["station_id", "ist_date"])[["station_id", "ist_date"] + fire_daily_columns]

for lag_days in [1, 2, 3]:
    lag_table = daily_fire.copy()
    # shifting the date forward puts day D's value on day D + lag_days
    lag_table["ist_date"] = lag_table["ist_date"] + pd.Timedelta(days=lag_days)
    lag_table = lag_table.rename(columns={
        "fire_frp_sum": "fire_frp_lag" + str(lag_days),
        "fire_count": "fire_count_lag" + str(lag_days),
    })
    df = df.merge(lag_table, on=["station_id", "ist_date"], how="left")

df = df.sort_values(["station_id", "timestamp"]).reset_index(drop=True)

# NaN lags = first days of 2018 or a missing previous day, LightGBM takes NaN natively
fire_source_columns = [c for c in df.columns if c.startswith("fire_frp") or c.startswith("fire_count")]
print(df[fire_source_columns].isna().sum())
assert (df[fire_source_columns].min() >= 0).all()

# log1p because FRP and counts are very skewed
df["fire_frp_today_log"] = np.log1p(df["fire_frp_sum"])
df["fire_frp_past3_log"] = np.log1p((df["fire_frp_lag1"] + df["fire_frp_lag2"] + df["fire_frp_lag3"]) / 3)
df["fire_count_today_log"] = np.log1p(df["fire_count"])
df["fire_count_past3_log"] = np.log1p((df["fire_count_lag1"] + df["fire_count_lag2"] + df["fire_count_lag3"]) / 3)

fire_count             0
fire_frp_sum           0
fire_frp_lag1       5267
fire_count_lag1     5267
fire_frp_lag2      13129
fire_count_lag2    13129
fire_frp_lag3      17253
fire_count_lag3    17253
dtype: int64


In [10]:
LAG_HOURS = [1, 3, 6]
NEIGHBOR_K = 3

station_list = sorted(df["station_id"].unique())
n_stations = len(station_list)
station_to_idx = {station: i for i, station in enumerate(station_list)}
target_idx = df["station_id"].map(station_to_idx).to_numpy()
n_rows = len(df)

# complete hourly index, so shifting the grid by n rows is exactly n hours
full_index = pd.date_range(df["timestamp"].min(), df["timestamp"].max(), freq="h")

# row -> position of its hour in the grid
time_idx = full_index.get_indexer(df["timestamp"])
assert (time_idx >= 0).all()

# pm25_grid[t, i] = pm25 of station i at hour t, NaN where the station has no row
pm25_grid = (
    df.pivot(index="timestamp", columns="station_id", values="pm25")
      .reindex(index=full_index, columns=station_list)
      .to_numpy(dtype=np.float32)
)

# value_grids[0] = pm25 now, then 1h, 3h, 6h ago
value_grids = [pm25_grid]
for lag in LAG_HOURS:
    shifted = np.full_like(pm25_grid, np.nan)
    shifted[lag:] = pm25_grid[:-lag]
    # a lagged value only counts where the station also has a row at hour t,
    # same as merging lag columns onto existing rows in the shared notebook
    shifted[np.isnan(pm25_grid)] = np.nan
    value_grids.append(shifted)

# availability per grid and number of stations reporting per hour, used for the comparable-row rule
avail_grids = [~np.isnan(g) for g in value_grids]
avail_count = [a.sum(axis=1) for a in avail_grids]

# trailing 24h mean of each station's pm25, current hour included, never future hours.
# grid rows are consecutive hours, so rolling(24) is exactly 24 hours
ROLL_HOURS = 24
roll_grid = (
    pd.DataFrame(pm25_grid)
    .rolling(ROLL_HOURS, min_periods=ROLL_HOURS // 2)   # needs at least 12 real hours in the window
    .mean()
    .to_numpy(dtype=np.float32)
)

print(n_stations, n_rows, pm25_grid.shape)

33 2392598 (76680, 33)


In [11]:
station_coords = df.drop_duplicates("station_id").set_index("station_id").loc[station_list, ["latitude", "longitude"]]
lat_r = np.radians(station_coords["latitude"].to_numpy())
lon_r = np.radians(station_coords["longitude"].to_numpy())

# dist_matrix[j, i] = haversine km from source station i to target station j (broadcast, no loop)
a_term = (np.sin((lat_r[None, :] - lat_r[:, None]) / 2) ** 2
          + np.cos(lat_r[:, None]) * np.cos(lat_r[None, :]) * np.sin((lon_r[None, :] - lon_r[:, None]) / 2) ** 2)
dist_matrix = 2 * 6371.0 * np.arcsin(np.sqrt(a_term))

# (n_rows, n_stations) distance from each row's station to every source, built once and reused by every fold.
# inf marks sources that can never be a neighbour for that row
dist_avail = dist_matrix[target_idx].astype(np.float32)
dist_avail[np.arange(n_rows), target_idx] = np.inf   # own station
dist_avail[~avail_grids[0][time_idx]] = np.inf       # source has no pm25 at that hour

In [12]:
IDW_POWER = 1.5       # K(distance) = distance ** -IDW_POWER
WIND_KAPPA = 1.0      # G(alignment) = exp(WIND_KAPPA * alignment), upwind weighs e**2 times more than downwind
MIN_DIST_KM = 0.5     # floor, so stations with near-identical coordinates cannot get infinite weight

# initial great-circle bearing from target station j to source station i, radians clockwise from north.
# axis 0 = target, axis 1 = source, same convention as dist_matrix
d_lon = lon_r[None, :] - lon_r[:, None]
bearing = np.arctan2(
    np.sin(d_lon) * np.cos(lat_r[None, :]),
    np.cos(lat_r[:, None]) * np.sin(lat_r[None, :])
    - np.sin(lat_r[:, None]) * np.cos(lat_r[None, :]) * np.cos(d_lon),
)
bearing_sin = np.sin(bearing).astype(np.float32)
bearing_cos = np.cos(bearing).astype(np.float32)

# wind_dir is the direction the wind comes FROM, so a source lying in that direction is upwind.
# alignment = cos(bearing - wind_from) = sin*sin + cos*cos, +1 directly upwind, -1 directly downwind.
# fancy index bearing_sin[target_idx] gives one (n_stations,) row per df row -> (n_rows, n_stations)
wind_sin = df["wind_dir_sin"].to_numpy(np.float32)[:, None]
wind_cos = df["wind_dir_cos"].to_numpy(np.float32)[:, None]
alignment = wind_sin * bearing_sin[target_idx] + wind_cos * bearing_cos[target_idx]

# w_i = K(distance_i) * G(alignment_i), for every source station at every row
dist_floor = np.maximum(dist_matrix, MIN_DIST_KM).astype(np.float32)
w_static = np.exp(WIND_KAPPA * alignment) / dist_floor[target_idx] ** IDW_POWER
del alignment

# a source with no pm25 at that hour or the row's own station gets zero weight
w_static[~avail_grids[0][time_idx]] = 0.0
w_static[np.arange(n_rows), target_idx] = 0.0
assert (w_static[np.arange(n_rows), target_idx] == 0).all()

# source pm25 at each row's hour, NaN zero-filled (safe, those weights are already 0)
pm25_rows = np.nan_to_num(value_grids[0][time_idx], nan=0.0)

# full-network weighted sums in float64. a fold later subtracts the held-out station's column
# instead of recomputing the (n_rows, n_stations) products 33 times
wsum_full = w_static.sum(axis=1, dtype=np.float64)
wpm_full = (w_static * pm25_rows).sum(axis=1, dtype=np.float64)

In [13]:
def build_neighbor_features(held_idx=None):

    dist_rows = dist_avail.copy()

    # the held-out station is removed from the network for every row, train and test
    if held_idx is not None:
        dist_rows[:, held_idx] = np.inf

    # K smallest distances per row via argpartition (no full sort). the order inside the K is arbitrary,
    # so "nearest" is found explicitly below, and mean/min/max/std do not depend on order
    order = np.argpartition(dist_rows, NEIGHBOR_K - 1, axis=1)[:, :NEIGHBOR_K]
    sel_dist = np.take_along_axis(dist_rows, order, axis=1)
    is_valid = np.isfinite(sel_dist)

    # a valid pick can never be the row's own station or the held-out one
    assert not np.any((order == target_idx[:, None]) & is_valid)
    if held_idx is not None:
        assert not np.any((order == held_idx) & is_valid)

    n_valid = is_valid.sum(axis=1)
    time_col = time_idx[:, None]     # (n_rows, 1), broadcasts against order (n_rows, K) to gather grid[t, station]
    row_col = target_idx[:, None]
    result = {}

    with warnings.catch_warnings():
        # all-NaN rows are expected, they just become NaN features
        warnings.simplefilter("ignore", category=RuntimeWarning)

        # pm25 now: mean, min, max, spread of the selected neighbours
        now_vals = value_grids[0][time_col, order].astype(np.float64)
        now_vals[~is_valid] = np.nan

        result["neighbor_mean"] = np.nanmean(now_vals, axis=1)
        result["neighbor_min"] = np.nanmin(now_vals, axis=1)
        result["neighbor_max"] = np.nanmax(now_vals, axis=1)

        # std needs at least 2 neighbours
        neighbor_std = np.nanstd(now_vals, axis=1)
        neighbor_std[n_valid < 2] = np.nan
        result["neighbor_std"] = neighbor_std

        # same neighbours, pm25 1h, 3h, 6h ago
        lag_names = ["neighbor_mean_lag1", "neighbor_mean_lag3", "neighbor_mean_lag6"]
        lag_arrays = []
        for k in range(3):
            lag_vals = value_grids[k + 1][time_col, order].astype(np.float64)
            lag_vals[~is_valid] = np.nan
            lag_arrays.append(lag_vals)
            result[lag_names[k]] = np.nanmean(lag_vals, axis=1)

        # recent change: per-neighbour difference first, then the mean, so only neighbours that have
        # BOTH values count (a difference of two means over different neighbour sets would be wrong)
        result["neighbor_change_1h"] = np.nanmean(now_vals - lag_arrays[0], axis=1)
        result["neighbor_change_3h"] = np.nanmean(now_vals - lag_arrays[1], axis=1)

        # trailing 24h mean of the same neighbours
        roll_vals = roll_grid[time_col, order].astype(np.float64)
        roll_vals[~is_valid] = np.nan
        result["neighbor_mean_roll24"] = np.nanmean(roll_vals, axis=1)

        # distance from the target to its neighbours
        dist_vals = sel_dist.astype(np.float64)
        dist_vals[~is_valid] = np.nan
        result["nearest_dist_km"] = np.nanmin(dist_vals, axis=1)
        result["mean_dist_km"] = np.nanmean(dist_vals, axis=1)

        # bearing from the target to its neighbours, (n_rows, K) gather from the (target, source) matrices
        b_sin = bearing_sin[row_col, order].astype(np.float64)
        b_cos = bearing_cos[row_col, order].astype(np.float64)
        b_sin[~is_valid] = np.nan
        b_cos[~is_valid] = np.nan

        # mean of unit vectors = resultant vector: direction says where the neighbours sit,
        # length (shorter when they surround the target) says how spread out they are
        result["mean_bearing_sin"] = np.nanmean(b_sin, axis=1)
        result["mean_bearing_cos"] = np.nanmean(b_cos, axis=1)

        # nearest neighbour's own bearing. argmin over sel_dist picks it (inf for invalid picks)
        nearest_pos = np.argmin(sel_dist, axis=1)
        rows_range = np.arange(n_rows)
        result["nearest_bearing_sin"] = b_sin[rows_range, nearest_pos]
        result["nearest_bearing_cos"] = b_cos[rows_range, nearest_pos]

        # wind-weighted mean over ALL available stations, normalised by the weight sum.
        # held-out station's contribution is subtracted from the precomputed full-network sums
        if held_idx is None:
            w_den = wsum_full
            w_num = wpm_full
        else:
            w_den = wsum_full - w_static[:, held_idx].astype(np.float64)
            w_num = wpm_full - (w_static[:, held_idx] * pm25_rows[:, held_idx]).astype(np.float64)

        # if only the held-out station had weight, the residual is rounding noise, so it becomes NaN
        w_ok = w_den > 1e-6
        result["neighbor_wmean"] = np.where(w_ok, w_num / np.where(w_ok, w_den, 1.0), np.nan)

    return result

# quick test, hold out station 0 and look at the missing counts
test_feats = build_neighbor_features(0)
for name in test_feats:
    print(name, np.isnan(test_feats[name]).sum())

neighbor_mean 0
neighbor_min 0
neighbor_max 0
neighbor_std 0
neighbor_mean_lag1 121
neighbor_mean_lag3 423
neighbor_mean_lag6 890
neighbor_change_1h 121
neighbor_change_3h 423
neighbor_mean_roll24 1154
nearest_dist_km 0
mean_dist_km 0
mean_bearing_sin 0
mean_bearing_cos 0
nearest_bearing_sin 0
nearest_bearing_cos 0
neighbor_wmean 0


In [14]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek

# Delhi seasons from month: winter 12-2, summer 3-6, monsoon 7-9, post-monsoon 10-11
df["season_winter"] = df["month"].isin([12, 1, 2]).astype(int)
df["season_summer"] = df["month"].between(3, 6).astype(int)
df["season_monsoon"] = df["month"].between(7, 9).astype(int)
df["season_post_monsoon"] = df["month"].isin([10, 11]).astype(int)

base_feature_columns = [
    "wind_speed_m_s", "wind_dir_sin", "wind_dir_cos", "boundary_layer_height_m",
    "hour", "day_of_week",
    "season_winter", "season_summer", "season_monsoon", "season_post_monsoon",
    "latitude", "longitude"
]
neighbor_feature_columns = [
    "neighbor_mean", "neighbor_min", "neighbor_max", "neighbor_std",
    "neighbor_mean_lag1", "neighbor_mean_lag3", "neighbor_mean_lag6",
    "neighbor_mean_roll24", "neighbor_change_1h", "neighbor_change_3h",
    "neighbor_wmean",
    "nearest_dist_km", "mean_dist_km",
    "nearest_bearing_sin", "nearest_bearing_cos", "mean_bearing_sin", "mean_bearing_cos"
]
feature_columns = base_feature_columns + neighbor_feature_columns

# fire columns stay LAST, the no-fire model is the first n_base_features columns of X_all
fire_feature_columns = ["fire_frp_today_log", "fire_frp_past3_log", "fire_count_today_log", "fire_count_past3_log"]
feature_columns_fire = feature_columns + fire_feature_columns

# the fire version is the no-fire columns plus 2 at the end, so no-fire = first n_base_features columns
n_base_features = len(feature_columns)

# pm25_original is the real target, pm25 (interpolated) is only ever an input
target_column = "pm25_original"

# boundary_layer_height_m is null for 2024-01-01 to 2024-06-30, so those rows are excluded here (same rule as the shared notebook)
required_columns = ["pm25_original", "boundary_layer_height_m", "wind_speed_m_s", "wind_dir_sin", "wind_dir_cos"]
row_ok = df[required_columns].notna().all(axis=1).to_numpy()

# one float32 matrix for all models. neighbour columns are placeholders, overwritten in place for every fold
X_all = np.empty((n_rows, len(feature_columns_fire)), dtype=np.float32)
for pos, col in enumerate(feature_columns_fire):
    if col not in neighbor_feature_columns:
        X_all[:, pos] = df[col].to_numpy()
y_all = df[target_column].to_numpy(dtype=np.float64)

neighbor_pos = {name: feature_columns_fire.index(name) for name in neighbor_feature_columns}

def set_neighbor_columns(held_idx):
    feats = build_neighbor_features(held_idx)
    for name, pos in neighbor_pos.items():
        X_all[:, pos] = feats[name]

# the GAM only predicts a row if some other station has pm25 at t, t-1h, t-3h and t-6h
def comparable_rows(held_idx):
    keep = np.ones(n_rows, dtype=bool)
    for avail, count in zip(avail_grids, avail_count):
        # stations other than the held-out one that have a value at each hour, then looked up per row
        other_sources = count - avail[:, held_idx].astype(np.int64)
        keep &= other_sources[time_idx] > 0
    return keep

print(len(feature_columns), len(feature_columns_fire), int(row_ok.sum()), n_rows)

29 33 2230464 2392598


In [18]:
# tuning uses neighbour features from the full network, held_idx=None
set_neighbor_columns(None)

# search only on every 6th hour (hourly data is heavily autocorrelated), the final LOSO fits use all rows
tune_mask = row_ok & (time_idx % 6 == 0)
X_tune = np.ascontiguousarray(X_all[tune_mask][:, :n_base_features])
y_tune = y_all[tune_mask]
groups_tune = target_idx[tune_mask]

# each CV fold keeps whole stations together
group_cv = GroupKFold(n_splits=3)

lgbm_param_grid = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [3, 4, 5, 6, 7, 8],
    "num_leaves": [15, 31, 47, 63],
    "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_samples": [10, 20, 30, 40]
}

# subsample_freq=1 is required, in LightGBM `subsample` does nothing while subsample_freq is 0.
# LightGBM already uses all cores per fit, so the search itself runs with n_jobs=1 to avoid oversubscription
lgbm_base_model = lgb.LGBMRegressor(
    random_state=42, verbosity=-1, n_jobs=-1, force_col_wise=True, subsample_freq=1
)

lgbm_random_search = RandomizedSearchCV(
    estimator=lgbm_base_model,
    param_distributions=lgbm_param_grid,
    n_iter=40,
    cv=group_cv,
    scoring="neg_mean_absolute_error",
    random_state=42,
    n_jobs=-1,
    refit=False
)

lgbm_random_search.fit(X_tune, y_tune, groups=groups_tune)

print("Best LightGBM params:", lgbm_random_search.best_params_)
print("Best grouped-CV MAE:", -lgbm_random_search.best_score_)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Best LightGBM params: {'subsample': 0.9, 'num_leaves': 15, 'n_estimators': 300, 'min_child_samples': 40, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
Best grouped-CV MAE: 23.890964242904264


In [19]:
def calculate_metrics(actual_values, predicted_values):

    # float64 so sums over millions of rows do not lose precision
    actual_values = np.asarray(actual_values, dtype=np.float64)
    predicted_values = np.asarray(predicted_values, dtype=np.float64)
    errors = actual_values - predicted_values

    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))

    ss_residual = np.sum(errors ** 2)
    ss_total = np.sum((actual_values - np.mean(actual_values)) ** 2)
    r2 = 1 - (ss_residual / ss_total)

    # positive means over-prediction
    bias = np.mean(predicted_values - actual_values)

    return mae, rmse, r2, bias

lgbm_best_params = lgbm_random_search.best_params_

def fit_and_predict(train_mask, test_mask):

    # boolean-mask indexing copies once per fold, both versions then slice columns from these copies
    X_train_full = X_all[train_mask]
    y_train = y_all[train_mask]
    X_test_full = X_all[test_mask]

    predictions = {}

    # same rows for both versions, only the number of columns differs
    for tag, n_cols in [("", n_base_features), ("_fire", len(feature_columns_fire))]:
        lgbm_model = lgb.LGBMRegressor(
            random_state=42, verbosity=-1, n_jobs=-1, force_col_wise=True, subsample_freq=1,
            **lgbm_best_params
        )
        # column slice is a non-contiguous view, make it contiguous once instead of letting LightGBM copy it
        lgbm_model.fit(np.ascontiguousarray(X_train_full[:, :n_cols]), y_train)
        predictions["lgbm" + tag + "_predicted"] = lgbm_model.predict(np.ascontiguousarray(X_test_full[:, :n_cols]))

    return predictions

timestamps = df["timestamp"].to_numpy()

# one DataFrame per fold, concatenated at the end (no per-row Python loop)
def make_result_frame(station_name, test_mask, predictions):
    frame = pd.DataFrame({
        "station_id": station_name,
        "timestamp": timestamps[test_mask],
        "actual": y_all[test_mask],
    })
    for model_name, values in predictions.items():
        frame[model_name] = values
    return frame

In [21]:
# time block for the date-leakage check: autumn stubble season (IST)
block_start = pd.Timestamp("2025-10-15")
block_end = pd.Timestamp("2025-12-01")
in_block = ((df["timestamp"] >= block_start) & (df["timestamp"] < block_end)).to_numpy()
print("rows inside the time block:", int(in_block.sum()))

main_frames = []
block_frames = []

for held_idx in range(n_stations):

    t_start = time.perf_counter()
    held_out_station = station_list[held_idx]

    # neighbours for every row come from the other stations only
    set_neighbor_columns(held_idx)

    is_held = target_idx == held_idx
    comparable = comparable_rows(held_idx)

    # standard LOSO: all hours of the other stations are available for training
    train_mask = row_ok & (~is_held)
    test_mask = row_ok & is_held & comparable

    if test_mask.sum() == 0:
        print(held_out_station, "skipped, no test rows")
        continue

    predictions = fit_and_predict(train_mask, test_mask)
    main_frames.append(make_result_frame(held_out_station, test_mask, predictions))

    # block LOSO: also remove the stubble-season hours of ALL stations from training
    block_train_mask = train_mask & (~in_block)
    block_test_mask = test_mask & in_block

    if block_test_mask.sum() > 0:
        block_predictions = fit_and_predict(block_train_mask, block_test_mask)
        block_frames.append(make_result_frame(held_out_station, block_test_mask, block_predictions))

    print(held_out_station, int(test_mask.sum()), int(block_test_mask.sum()), f"{time.perf_counter() - t_start:.0f}s")

loso_df = pd.concat(main_frames, ignore_index=True)
block_df = pd.concat(block_frames, ignore_index=True)

loso_df.to_csv("/kaggle/working/loso_df_lgbm_fire_33station.csv", index=False)
block_df.to_csv("/kaggle/working/block_df_lgbm_fire_33station.csv", index=False)

print(loso_df.shape, block_df.shape)

rows inside the time block: 36723
alipur 61510 1106 159s
anand_vihar 65681 1117 150s
bawana 65523 1119 148s
crri_mathura_road 68115 1111 148s
dr_karni_singh_shooting_range 66524 1124 148s
ito 66689 1102 147s
jahangirpuri 68557 1101 149s
jawaharlal_nehru_stadium 68608 1103 149s
major_dhyan_chand_national_stadium 69664 1117 149s
mandir_marg 67385 1094 148s
mundka 65579 1111 149s
najafgarh 67058 1029 150s
narela 69499 1128 148s


KeyboardInterrupt: 

In [22]:
model_columns = ["lgbm_predicted", "lgbm_fire_predicted"]

for col in model_columns:
    mae, rmse, r2, bias = calculate_metrics(loso_df["actual"], loso_df[col])
    print(f"{col:22s} MAE {mae:.3f}  RMSE {rmse:.2f}  R2 {r2:.3f}  bias {bias:.2f}")

per_station_rows = []

for station, station_rows in loso_df.groupby("station_id"):

    row = {"station_id": station, "n_rows": len(station_rows)}

    for col in model_columns:
        tag = col.replace("_predicted", "")
        mae, rmse, r2, bias = calculate_metrics(station_rows["actual"], station_rows[col])
        row[tag + "_mae"] = mae
        row[tag + "_rmse"] = rmse
        row[tag + "_r2"] = r2
        row[tag + "_bias"] = bias

    per_station_rows.append(row)

per_station_df = pd.DataFrame(per_station_rows)
print(per_station_df.round(3).to_string())

print("fire lowers MAE at", int((per_station_df["lgbm_fire_mae"] < per_station_df["lgbm_mae"]).sum()),
      "of", len(per_station_df), "stations")

lgbm_predicted         MAE 24.734  RMSE 43.00  R2 0.808  bias 1.42
lgbm_fire_predicted    MAE 24.702  RMSE 42.98  R2 0.808  bias 1.17
                            station_id  n_rows  lgbm_mae  lgbm_rmse  lgbm_r2  lgbm_bias  lgbm_fire_mae  lgbm_fire_rmse  lgbm_fire_r2  lgbm_fire_bias
0                               alipur   61510    20.651     35.020    0.867      9.364         20.624          35.380         0.864           9.114
1                          anand_vihar   65681    27.529     46.840    0.841    -18.314         28.059          47.664         0.836         -20.003
2                          ashok_vihar   68243    23.765     37.634    0.878     17.341         24.104          37.820         0.877          17.747
3                       aurobindo_marg   66207    18.018     31.047    0.844     11.019         17.760          30.702         0.848          10.562
4                            aya_nagar   66874    25.200     39.848    0.709      9.241         24.608          39.427   

In [23]:
ts = loso_df["timestamp"]
window_mask = (ts >= block_start) & (ts < block_end)

subsets = [
    ("all rows", loso_df),
    ("actual < 100", loso_df[loso_df["actual"] < 100]),
    ("actual >= 100", loso_df[loso_df["actual"] >= 100]),
    ("Oct 15-Nov 30 2025, normal LOSO", loso_df[window_mask]),
    ("Oct 15-Nov 30 2025, window removed from training", block_df)
]

for label, data in subsets:
    print("-----", label, "| n =", len(data))
    for col in model_columns:
        mae, rmse, r2, bias = calculate_metrics(data["actual"], data[col])
        print(f"{col:22s} MAE {mae:.3f}  RMSE {rmse:.2f}  R2 {r2:.3f}  bias {bias:.2f}")

----- all rows | n = 2229994
lgbm_predicted         MAE 24.734  RMSE 43.00  R2 0.808  bias 1.42
lgbm_fire_predicted    MAE 24.702  RMSE 42.98  R2 0.808  bias 1.17
----- actual < 100 | n = 1441564
lgbm_predicted         MAE 14.921  RMSE 23.71  R2 0.041  bias 7.11
lgbm_fire_predicted    MAE 14.822  RMSE 23.59  R2 0.051  bias 6.83
----- actual >= 100 | n = 788430
lgbm_predicted         MAE 42.677  RMSE 64.82  R2 0.604  bias -9.00
lgbm_fire_predicted    MAE 42.768  RMSE 64.87  R2 0.603  bias -9.16
----- Oct 15-Nov 30 2025, normal LOSO | n = 36426
lgbm_predicted         MAE 43.936  RMSE 69.23  R2 0.606  bias 3.28
lgbm_fire_predicted    MAE 43.895  RMSE 69.02  R2 0.609  bias 2.30
----- Oct 15-Nov 30 2025, window removed from training | n = 36426
lgbm_predicted         MAE 43.912  RMSE 69.17  R2 0.607  bias 2.75
lgbm_fire_predicted    MAE 43.816  RMSE 68.85  R2 0.611  bias 2.41


In [24]:
loso_df["lgbm_sq_error"] = (loso_df["actual"] - loso_df["lgbm_predicted"]) ** 2

total_sq_error = loso_df["lgbm_sq_error"].sum()
total_rows = len(loso_df)

share_df = loso_df.groupby("station_id")["lgbm_sq_error"].agg(["size", "sum"]).reset_index()
share_df["pct_of_rows"] = 100 * share_df["size"] / total_rows
share_df["pct_of_sq_error"] = 100 * share_df["sum"] / total_sq_error
share_df = share_df.drop(columns="sum").rename(columns={"size": "n_rows"})

print(share_df.sort_values("pct_of_sq_error", ascending=False).round(2).to_string())

                            station_id  n_rows  pct_of_rows  pct_of_sq_error
10                ihbas_dilshad_garden   67578         3.03             8.69
8                                  dtu   66459         2.98             5.94
28                            shadipur   68763         3.08             5.58
21                                nsut   69256         3.11             5.14
19                         nehru_nagar   69743         3.13             5.02
12                        jahangirpuri   68557         3.07             4.76
24                        punjabi_bagh   68883         3.09             4.10
17                           najafgarh   67058         3.01             4.02
16                              mundka   65579         2.94             3.76
1                          anand_vihar   65681         2.95             3.49
11                                 ito   66689         2.99             3.47
26                            rk_puram   67169         3.01             3.27